### Import Libraries dan Load Data

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
from time import perf_counter
import json

# Sampling methods
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
import imblearn

print("imblearn version:", imblearn.__version__)
print("Starting sequential sampling process...")

# Load original standardized data
print("\nLoading original standardized training dataset...")
train_data = pd.read_csv("../dataset_splitted/train_split_scaled.csv")

print(f"Loaded dataset shape: {train_data.shape}")

# Separate features and target
target_col = "category"
X_original = train_data.drop(columns=[target_col])
y_original = train_data[target_col]

print(f"\nOriginal data:")
print(f"X shape: {X_original.shape}")
print(f"y shape: {y_original.shape}")

# Get class info
unique_classes = sorted(y_original.unique())
n_classes = len(unique_classes)
original_distribution = Counter(y_original)

print(f"\nClasses: {n_classes}")
print(f"Original class distribution:")
for class_label, count in sorted(original_distribution.items()):
    print(f"  Class {class_label}: {count:,}")
    
print(f"Total samples: {len(y_original):,}")

### Fungsi Helper untuk Analisis dan Penyimpanan

In [ ]:
def analyze_distribution(y, stage_name, previous_distribution=None):
    """Analyze and display class distribution"""
    distribution = Counter(y)
    total_samples = len(y)
    
    print(f"\n{'='*60}")
    print(f"{stage_name.upper()} - CLASS DISTRIBUTION ANALYSIS")
    print(f"{'='*60}")
    
    print(f"Total samples: {total_samples:,}")
    print(f"Number of classes: {len(distribution)}")
    
    print(f"\nClass distribution:")
    for class_label, count in sorted(distribution.items()):
        percentage = (count / total_samples) * 100
        print(f"  Class {class_label}: {count:,} ({percentage:.2f}%)")
    
    # Balance metrics
    min_samples = min(distribution.values())
    max_samples = max(distribution.values())
    avg_samples = total_samples / len(distribution)
    balance_ratio = min_samples / max_samples
    
    print(f"\nBalance metrics:")
    print(f"  Min samples: {min_samples:,}")
    print(f"  Max samples: {max_samples:,}")
    print(f"  Avg samples: {avg_samples:,.0f}")
    print(f"  Balance ratio (min/max): {balance_ratio:.3f}")
    
    # Changes from previous stage
    if previous_distribution is not None:
        prev_total = sum(previous_distribution.values())
        change = total_samples - prev_total
        change_pct = (change / prev_total) * 100
        
        print(f"\nChanges from previous stage:")
        print(f"  Sample change: {change:+,} ({change_pct:+.2f}%)")
        
        print(f"  Per-class changes:")
        for class_label in sorted(unique_classes):
            prev_count = previous_distribution.get(class_label, 0)
            curr_count = distribution.get(class_label, 0)
            class_change = curr_count - prev_count
            class_change_pct = (class_change / prev_count * 100) if prev_count > 0 else 0
            print(f"    Class {class_label}: {prev_count:,} → {curr_count:,} ({class_change:+,}, {class_change_pct:+.1f}%)")
    
    return distribution

def save_data_and_metadata(X, y, stage_name, distribution, metadata_extra=None):
    """Save data and metadata for a specific stage"""
    print(f"\nSaving {stage_name} data...")
    
    # Convert to DataFrame
    X_df = pd.DataFrame(X, columns=X_original.columns)
    y_df = pd.Series(y, name=target_col)
    combined_data = pd.concat([X_df, y_df], axis=1)
    
    # Save data
    data_path = f"train_split_scaled_{stage_name.lower().replace(' ', '_').replace('-', '_')}.csv"
    combined_data.to_csv(data_path, index=False)
    print(f"Data saved to: {data_path}")
    
    # Create metadata
    metadata = {
        "stage": stage_name,
        "timestamp": pd.Timestamp.now().isoformat(),
        "total_samples": int(len(y)),
        "n_features": int(X.shape[1]),
        "n_classes": int(len(distribution)),
        "class_distribution": {str(k): int(v) for k, v in distribution.items()},
        "balance_ratio": float(min(distribution.values()) / max(distribution.values())),
        "imblearn_version": imblearn.__version__
    }
    
    if metadata_extra:
        # Simple update without complex type conversion
        metadata.update(metadata_extra)
    
    # Save metadata
    metadata_path = f"metadata_{stage_name.lower().replace(' ', '_').replace('-', '_')}.json"
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    print(f"Metadata saved to: {metadata_path}")
    
    return data_path, metadata_path

print("Helper functions defined successfully!")

### Stage 0: Original Data Analysis

In [ ]:
# Analyze original distribution
original_dist = analyze_distribution(y_original, "Original Data")

# Store processing times
processing_times = {}

### Stage 1: Random Under Sampler (RUS)

Mengurangi kelas mayoritas untuk mendekati keseimbangan.

In [ ]:
print("\n" + "="*70)
print("STAGE 1: RANDOM UNDER SAMPLER")
print("="*70)

start_time = perf_counter()
target_samples_per_class = 500_000

print(f"Target samples per class: {target_samples_per_class:,}")

rus_strategy = {}

for class_label in unique_classes:
    original_count = original_dist[class_label]
    # Use the calculated target per class, but don't exceed original count for small classes
    rus_strategy[class_label] = min(target_samples_per_class, original_count)

print(f"RUS sampling strategy (target per class: {target_samples_per_class:,}):")
for class_label, target in rus_strategy.items():
    original_count = original_dist[class_label]
    print(f"  Class {class_label}: {original_count:,} → {target:,}")

# Apply RUS
rus_params = {
    'sampling_strategy': rus_strategy,
    'random_state': 42
}

rus = RandomUnderSampler(**rus_params)
print(f"\nApplying Random Under Sampler...")
X_rus, y_rus = rus.fit_resample(X_original, y_original)

end_time = perf_counter()
rus_time = end_time - start_time
processing_times['RUS'] = rus_time

print(f"RUS completed in {rus_time:.2f} seconds")

# Analyze RUS results
rus_dist = analyze_distribution(y_rus, "After RUS", original_dist)

# Save RUS data
rus_metadata = {
    'method': 'RandomUnderSampler',
    'parameters': {
        'sampling_strategy': {str(k): int(v) for k, v in rus_strategy.items()},
        'random_state': 42
    },
    'target_strategy': 'equal_distribution',
    'target_samples_per_class': int(target_samples_per_class),
    'processing_time_seconds': float(rus_time)
}

save_data_and_metadata(X_rus, y_rus, "RUS", rus_dist, rus_metadata)

### Stage 2: SMOTE (Synthetic Minority Oversampling)

Menambah synthetic samples untuk kelas minoritas.

In [ ]:
print("\n" + "="*70)
print("STAGE 2: SMOTE (SYNTHETIC MINORITY OVERSAMPLING)")
print("="*70)

start_time = perf_counter()

# Apply SMOTE
smote_params = {
    'sampling_strategy': 'auto',
    'k_neighbors': 5,
    'random_state': 42
}

smote = SMOTE(**smote_params)
print(f"\nApplying SMOTE...")
X_smote, y_smote = smote.fit_resample(X_rus, y_rus)

end_time = perf_counter()
smote_time = end_time - start_time
processing_times['SMOTE'] = smote_time

print(f"SMOTE completed in {smote_time:.2f} seconds")

# Analyze SMOTE results
smote_dist = analyze_distribution(y_smote, "After SMOTE", rus_dist)

# Save SMOTE data
smote_metadata = {
    'method': 'SMOTE',
    'parameters': {
        'sampling_strategy': 'auto',
        'k_neighbors': 5,
        'random_state': 42
    },
    'processing_time_seconds': float(smote_time)
}

save_data_and_metadata(X_smote, y_smote, "RUS-SMOTE", smote_dist, smote_metadata)